In [31]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

In [32]:
spark = SparkSession.builder.appName("SparkPartition").getOrCreate()

In [33]:
df = spark.range(0, 10000000).withColumn("value", col("id") % 1000)

In [34]:
print("Before Parition :",df.rdd.getNumPartitions())

Before Parition : 2


In [24]:
df.write.mode("overwrite").csv("Output/before_partition_data",header=True)

In [35]:
df_rep = df.repartition(20)

In [36]:
print("After Partitions:", df_rep.rdd.getNumPartitions())

[Stage 8:>                                                          (0 + 2) / 2]

After Partitions: 20


In [37]:
df_rep.write.mode("overwrite").csv("Output/after_partition_data",header=True)

In [38]:
df_coalesced = df_rep.coalesce(5)

In [39]:
df_coalesced.write.mode("overwrite").csv("Output/after_coalesce_data",header=True)

In [57]:
import time

In [58]:
optimized_df = df.filter(col("value")>500).filter(col("id")<5000000).select("id","value")

In [59]:
optimized_df.explain()

== Physical Plan ==
*(1) Project [id#21L, (id#21L % 1000) AS value#23L]
+- *(1) Filter (((id#21L % 1000) > 500) AND (id#21L < 5000000))
   +- *(1) Range (0, 10000000, step=1, splits=2)




In [60]:
start_time= time.time()
count_uncached = optimized_df.count()
end_time = time.time()
print(f"1. optimize execution | count : {count_uncached} | Time :{round(end_time-start_time,4)} seconds" )

1. optimize execution | count : 2495000 | Time :0.1129 seconds


In [61]:
optimized_df.cache()

DataFrame[id: bigint, value: bigint]

In [62]:
start_time= time.time()
count_cached = optimized_df.count()
end_time = time.time()
print(f"2. execution cached | count : {count_cached} | Time :{round(end_time-start_time,4)} seconds" )

[Stage 41:=============================>                            (1 + 1) / 2]

2. execution cached | count : 2495000 | Time :1.1443 seconds


In [63]:
start_time= time.time()
count_cached = optimized_df.count()
end_time = time.time()
print(f"3. execution cached | count : {count_cached} | Time :{round(end_time-start_time,4)} seconds" )

3. execution cached | count : 2495000 | Time :0.111 seconds


In [64]:
optimized_df.unpersist()

DataFrame[id: bigint, value: bigint]